Exercises:
E01: train a trigram language model, i.e. take two characters as an input to predict the 3rd one. Feel free to use either counting or a neural net. Evaluate the loss; Did it improve over a bigram model?


E02: split up the dataset randomly into 80% train set, 10% dev set, 10% test set. Train the bigram and trigram models only on the training set. Evaluate them on dev and test splits. What can you see?


E03: use the dev set to tune the strength of smoothing (or regularization) for the trigram model - i.e. try many possibilities and see which one works best based on the dev set loss. What patterns can you see in the train and dev set loss as you tune this strength? Take the best setting of the smoothing and evaluate on the test set once and at the end. How good of a loss do you achieve?


E04: we saw that our 1-hot vectors merely select a row of W, so producing these vectors explicitly feels wasteful. Can you delete our use of F.one_hot in favor of simply indexing into rows of W?


E05: look up and use F.cross_entropy instead. You should achieve the same result. Can you think of why we'd prefer to use F.cross_entropy instead?


E06: meta-exercise! Think of a fun/interesting exercise and complete it.

### Trigram Model

In [445]:
names = open('names.txt', 'r').read().splitlines() 

In [446]:
tri_model = {}
for w in names:
    w = ['.'] + list(w) + ['.']
    for w, w1, w2 in zip(w, w[1:], w[2:]):
        trigram = (w, w1, w2)
        tri_model[trigram] = tri_model.get(trigram, 0) + 1

In [447]:
len(tri_model)

6037

In [448]:
max(tri_model, key=tri_model.get), tri_model[max(tri_model, key=tri_model.get)]

(('a', 'h', '.'), 1714)

In [449]:
min(tri_model, key=tri_model.get), tri_model[min(tri_model, key=tri_model.get)]

(('n', 'h', 'o'), 1)

In [450]:
chars = list(sorted(set(''.join(names))))
chars.insert(0, '.')

In [451]:
stoi = {s:i for i, s in enumerate(chars)}
stoi['.'], stoi['z']

(0, 26)

In [452]:
sstoi = {}
idx = 0
for char1 in chars: 
    for char2 in chars: 
        sstoi[(char1, char2)] = idx
        idx += 1

In [453]:
len(sstoi)

729

In [454]:
len(stoi) 

27

In [455]:
import torch
N = torch.zeros(27*27, 27, dtype=torch.int)

In [456]:
N.shape

torch.Size([729, 27])

In [457]:
# Create a count matrix
for w in names:
    w = ['.'] + list(w) + ['.']
    for w, w1, w2 in zip(w, w[1:], w[2:]):
        idx = sstoi[(w, w1)]
        target_index = stoi[w2]
        N[idx, target_index] += 1

In [458]:
max(tri_model, key=tri_model.get), tri_model[max(tri_model, key=tri_model.get)]

(('a', 'h', '.'), 1714)

In [459]:
N[sstoi[('a', 'h')]] # It was a check if the model is performing correctly 

tensor([1714,  111,    3,    2,   11,   46,    1,    0,    1,  101,    7,   15,
          73,   88,   57,    9,    1,    1,   34,   17,    1,    8,    5,    1,
           0,   11,   14], dtype=torch.int32)

In [460]:
N[0], sstoi[('.', '.')]

(tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0], dtype=torch.int32),
 0)

In [461]:
N.shape

torch.Size([729, 27])

In [462]:
P = N / N.sum(1, keepdims=True) # Probability Matrix

In [463]:
P.sum(1)[:3]

tensor([nan, 1., 1.])

It is working correctly since we have a sum of one

### Now we sample

In [464]:
itos = {i:s for i,s in enumerate(stoi)}
iitos = {i: s for i, s in enumerate(sstoi)} 

In [465]:
len(itos), len(iitos)

(27, 729)

In [466]:
g = torch.Generator().manual_seed(2147483647)

In [468]:
sstoi[('.', 'z')], N[26]

(26,
 tensor([  0, 456,   0,   0,   0, 135,   0,   0,  33,  86,   0,   0,   3,   1,
           0,  72,   0,   0,   1,   2,   0,  47,   1,   0,   0,  91,   1],
        dtype=torch.int32))

In [469]:
result = list(filter(lambda item: item[1] == 456, tri_model.items()))
result

[(('.', 'z', 'a'), 456)]

In [470]:
n = N[:27]

In [471]:
n = n.sum(1) 

In [472]:
n = n / n.sum(0)

In [473]:
n # Probability for the starting character after '.'

tensor([0.0000, 0.1377, 0.0408, 0.0481, 0.0528, 0.0478, 0.0130, 0.0209, 0.0273,
        0.0184, 0.0756, 0.0925, 0.0491, 0.0792, 0.0358, 0.0123, 0.0161, 0.0029,
        0.0512, 0.0642, 0.0408, 0.0024, 0.0117, 0.0096, 0.0042, 0.0167, 0.0290])

In [474]:
ix = torch.multinomial(n, num_samples=1, replacement=True, generator=g).item()

In [475]:
ix

3

In [476]:
ch = iitos[ix]
''.join(ch)

'.c'

In [477]:
itos[ix]

'c'

In [479]:
P[ix]

tensor([0.0000, 0.4073, 0.0000, 0.0000, 0.0000, 0.0422, 0.0000, 0.0000, 0.2283,
        0.0285, 0.0013, 0.0000, 0.0441, 0.0000, 0.0000, 0.1654, 0.0000, 0.0000,
        0.0435, 0.0000, 0.0000, 0.0084, 0.0000, 0.0000, 0.0000, 0.0298, 0.0013])

In [506]:
for _ in range(5):
    out = []

    ix = torch.multinomial(n, num_samples=1, replacement=True, generator=g).item()
    c1, c2 = iitos[ix]
    out.extend([c1, c2])   

    while True:
        c3_ix = torch.multinomial(P[ix], num_samples=1, replacement=True, generator=g).item()
        c3= itos[c3_ix] 

        out.append(c3)   # append ONLY the new character

        if c3 == '.':
            break
            
        ix=sstoi[(c2, c3)]
        c2 = c3 # shift window


    print(''.join(out))


.iya.
.dyelian.
.za.
.fie.
.mck.


<b> Way better than the bigram model, this more natural, sound so much close to real names

### Measure loss of Trigram model using MLE

In [529]:
log_likelihood = 0.0
n = 0
for w in names[:1]:
    w = ['.'] + list(w) + ['.']
    for w, w1, w2 in zip(w, w[1:], w[2:]):
        idx = sstoi[(w, w1)]
        target_index = stoi[w2]
        prob = P[idx, target_index]
        log_prob = torch.log(prob)
        log_likelihood +=  log_prob
        n +=1
        print((w, w1), w2, f'{prob.item():.4f}, {log_prob:.2f}')

        
nll=-log_likelihood

('.', 'e') m 0.1881, -1.67
('e', 'm') m 0.1300, -2.04
('m', 'm') a 0.4286, -0.85
('m', 'a') . 0.0672, -2.70


<br><b>
    <p style="font-size:18px;"> <b> Now let's see the total loss for the dataset</b></p>

In [530]:
log_likelihood = 0.0
n = 0
for w in names:
    w = ['.'] + list(w) + ['.']
    for w, w1, w2 in zip(w, w[1:], w[2:]):
        idx = sstoi[(w, w1)]
        target_index = stoi[w2]
        prob = P[idx, target_index]
        log_prob = torch.log(prob)
        log_likelihood += log_prob
        n+=1

        
nll=-log_likelihood
print(nll/n) # normalize

tensor(2.0620)


<b> The bigram model had a loss of 2.51, so we are performing better<b>

### Building Neural Netwok one

<p style="font-size:18px;">We must identify a starting character, since in our bigram model our starting point were '.' this was our marking point.<br><br> It was clear, but here we have different options, we can't use '.', because our start is two characters followed by one character.  then we are sampling for character for that specific letter, so we denote starting by '.','.' so our model know this. after it sees '..' it will know it is a start of word<br><br> For any n-gram model, the start is n-1, big gram is 2-1, so our start is fixed '.', trigram is '.'.', so we should choose any special character that denotes a start, since '.' '.' exists in our itos and stoi, we can use it directly. 
</p



In [621]:
xs, ys = [], []


for w in names:
    w = ['.', '.'] + list(w) + ['.']
    for w, w1, w2 in zip(w, w[1:], w[2:]):
        idx = sstoi[(w, w1)]
        target_index = stoi[w2]
        xs.append(idx)
        ys.append(target_index)
xs = torch.tensor(xs)
ys = torch.tensor(ys)

In [622]:
len(xs), len(ys) # previous we had 32,000, since it trigram we have less examples

(228146, 228146)

In [623]:
import torch.nn.functional as F
xs_onehot = F.one_hot(xs, num_classes=27*27).float()
W = torch.randn(27*27,27, requires_grad=True, generator=g) 

In [624]:
num = xs.nelement()
num

228146

In [686]:
for i in range(1000): 
    logits = xs_onehot @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdims=True)
    loss = -probs[torch.arange(num), ys].log().mean()


    W.grad = None
    loss.backward()
    
    W.data -= 10 * W.grad

print(f"pass{i}, with loss {loss.item()}")    

pass999, with loss 2.3382675647735596


### Sample 

In [688]:
g = torch.Generator().manual_seed(2147483647)
for i in range(10): 
    out = []
    w, w1 = '.', '.'
    ix = sstoi[w, w1]
    
    while True: 
       
        xenc = F.one_hot(torch.tensor(ix), num_classes=27*27).float()

        logits = xenc @ W
      
        counts = logits.exp()

 
        probs = counts / counts.sum(0) 

        w2_ix = torch.multinomial(probs, num_samples=1, replacement=True, generator=g).item()
        w2 = itos[w2_ix]
        
        
        if w2=='.': 
            break
            
        out.append(w2)
        
        ix = sstoi[w1, w2]
        w1 = w2
     

        
        
            
    print(''.join(out))

    

cexbdczoglfurkrichityhkmsonimilea
noluwak
ka
da
samiyaubjtphi
gitai
mozarickxujkwpthda
kaley
masideu
niavion


### Train, Dev, Test Set

In [773]:
names = open('names.txt', 'r').read().splitlines() 

In [774]:
import random
import torch
import torch.nn.functional as F

random.seed(42)
g = torch.Generator().manual_seed(2147483647)
random.shuffle(names)


In [775]:

n = len(names)

n_train = int(0.8 * n)
n_dev   = int(0.1 * n)

n, n_train, n_dev

(32033, 25626, 3203)

In [776]:
def build_dataset(names):
    xs, ys = [], []
    for w in names:
        w = ['.', '.'] + list(w) + ['.']
        for c0, c1, c2 in zip(w, w[1:], w[2:]):
            xs.append(sstoi[(c0, c1)])
            ys.append(stoi[c2])
    return torch.tensor(xs), torch.tensor(ys)


In [777]:
train_names = names[:n_train]
dev_set = names[n_train:n_train + n_dev]
test_set = names[n_train+n_dev:]
len(x_train), len(dev_set), len(dev_set)

In [778]:
xs_train, ys_train = build_dataset(train_names)
xs_dev, ys_dev = build_dataset(dev_set)
xs_test, ys_test  = build_dataset(test_set)

In [750]:
xs_onehot = F.one_hot(xs_train, num_classes=27*27).float()
W = torch.randn(27*27,27, requires_grad=True, generator=g) 

In [755]:
for _ in range(10): 
    for i in range(100): 
        logits = xs_onehot @ W
        counts = logits.exp()
        probs = counts / counts.sum(1, keepdims=True)
        loss = -probs[torch.arange(num), ys_train].log().mean()


        W.grad = None
        loss.backward()

        W.data -= 10 * W.grad

    print(f"pass{_}, with loss {loss.item()}")   

pass0, with loss 2.3321573734283447
pass1, with loss 2.324296474456787
pass2, with loss 2.3172779083251953
pass3, with loss 2.31097149848938
pass4, with loss 2.3052713871002197
pass5, with loss 2.3000924587249756
pass6, with loss 2.2953646183013916
pass7, with loss 2.291030168533325
pass8, with loss 2.287041425704956
pass9, with loss 2.2833573818206787


In [840]:
def evaluate_trigram(W, xs, ys):
    xenc = F.one_hot(xs, num_classes=27*27).float()
    logits = xenc @ W
    probs = logits.softmax(dim=1)
    loss = -probs[torch.arange(len(xs)), ys].log().mean()
    return loss.item()


In [841]:
train_loss = evaluate_trigram(W, xs_train, ys_train)
dev_loss   = evaluate_trigram(W, x_dev, y_dev)
test_loss  = evaluate_trigram(W, x_test, y_test)

print(f"train: {train_loss:.3f}")
print(f"dev:   {dev_loss:.3f}")
print(f"test:  {test_loss:.3f}")


train: 3.412
dev:   3.259
test:  3.457


### Excercise 3, tune the model based on the devset

In [785]:
loss_models = {}

xs_train_onehot = F.one_hot(xs_train, num_classes=27*27).float()
num_train = xs_train.shape[0]

for reg_term in [0, 0.001, 0.01, 0.05, 0.1, 0.5, 1]:
    W = torch.randn(27*27, 27, requires_grad=True, generator=g)

    for _ in range(200):
        logits = xs_train_onehot @ W
        probs = logits.softmax(dim=1)

        loss = -probs[torch.arange(num_train), ys_train].log().mean() \
               + reg_term * (W**2).mean()

        W.grad = None
        loss.backward()
        W.data -= 50 * W.grad

    train_loss = evaluate_trigram(W, xs_train, ys_train)
    dev_loss   = evaluate_trigram(W, xs_dev, ys_dev)

    loss_models[reg_term] = dev_loss

    print(f"train_loss: {train_loss:.3f} | reg={reg_term}")
    print(f"dev_loss:   {dev_loss:.3f} | reg={reg_term}")
    print("-" * 40)

best_reg, best_loss = min(loss_models.items(), key=lambda x: x[1])
print(f"Best model: dev_loss={best_loss:.3f}, reg={best_reg}")


train_loss: 2.382 | reg=0
dev_loss:   2.392 | reg=0
----------------------------------------
train_loss: 2.381 | reg=0.001
dev_loss:   2.389 | reg=0.001
----------------------------------------
train_loss: 2.379 | reg=0.01
dev_loss:   2.387 | reg=0.01
----------------------------------------
train_loss: 2.381 | reg=0.05
dev_loss:   2.389 | reg=0.05
----------------------------------------
train_loss: 2.378 | reg=0.1
dev_loss:   2.388 | reg=0.1
----------------------------------------
train_loss: 2.393 | reg=0.5
dev_loss:   2.400 | reg=0.5
----------------------------------------
train_loss: 2.425 | reg=1
dev_loss:   2.430 | reg=1
----------------------------------------
Best model: dev_loss=2.387, reg=0.01


In [838]:

xs_train_onehot = F.one_hot(xs_train, num_classes=27*27).float()
num_train = xs_train.shape[0]
reg_term = 0.01
W = torch.randn(27*27, 27, requires_grad=True, generator=g)

for _ in range(100):
    
    counts = W.exp() 
    probs = counts / counts.sum(1, keepdims=True)

    loss = -probs[xs_train, ys_train].log().mean() \
           + reg_term * (W**2).mean()

    W.grad = None
    loss.backward()
    W.data -= 50 * W.grad
    

train_loss = evaluate_trigram(W, xs_train, ys_train)
dev_loss   = evaluate_trigram(W, xs_dev, ys_dev)

loss_models[reg_term] = dev_loss

print(f"train_loss: {train_loss:.3f} | reg={reg_term}")
print(f"dev_loss:   {dev_loss:.3f} | reg={reg_term}")
print("-" * 40)



train_loss: 2.502 | reg=0.01
dev_loss:   2.505 | reg=0.01
----------------------------------------


In [839]:
test_loss   = evaluate_trigram(W, xs_test, ys_test)
print(f"test_loss:   {test_loss:.3f} ")

test_loss:   2.508 


Same as test loss, which means no over fitting probably this is the best we can do

### Exercise 4: Remove One hot encoding with indexing directly 

In [818]:
print(xs_onehot[3] @ W)

tensor([ 0.0678,  1.5159, -0.0872, -0.0847, -0.0865,  0.2097, -0.0844, -0.0803,
        -0.0869,  0.0590, -0.0834, -0.0866, -0.0911, -0.0800, -0.0780, -0.0768,
        -0.0817, -0.0842, -0.0865, -0.0342, -0.0344, -0.0912, -0.0856, -0.0884,
        -0.0919, -0.0829, -0.0815], grad_fn=<SqueezeBackward4>)


In [817]:
W[xs_train[3], :]

tensor([ 0.0678,  1.5159, -0.0872, -0.0847, -0.0865,  0.2097, -0.0844, -0.0803,
        -0.0869,  0.0590, -0.0834, -0.0866, -0.0911, -0.0800, -0.0780, -0.0768,
        -0.0817, -0.0842, -0.0865, -0.0342, -0.0344, -0.0912, -0.0856, -0.0884,
        -0.0919, -0.0829, -0.0815], grad_fn=<SelectBackward0>)

In [850]:
def evaluate_trigram(W, xs, ys):
    logits = W[xs] 
    probs = logits.softmax(dim=1)
    loss = -probs[torch.arange(len(xs)), ys].log().mean()
    return loss.item()


In [851]:
num_train = xs_train.shape[0]
reg_term = 0.5
W = torch.randn(27*27, 27, requires_grad=True, generator=g)

In [853]:
for _ in range(100):
    
    logits = W[xs_train] 

    counts = logits.exp() 
    probs = counts / counts.sum(1, keepdims=True)

    loss = -probs[torch.arange(num_train), ys_train].log().mean() \
           + reg_term * (W**2).mean()

    W.grad = None
    loss.backward()
    W.data -= 50 * W.grad
    

train_loss = evaluate_trigram(W, xs_train, ys_train)
dev_loss   = evaluate_trigram(W, xs_dev, ys_dev)

loss_models[reg_term] = dev_loss

print(f"train_loss: {train_loss:.3f} | reg={reg_term}")
print(f"dev_loss:   {dev_loss:.3f} | reg={reg_term}")
print("-" * 40)



train_loss: 2.381 | reg=0.01
dev_loss:   2.396 | reg=0.01
----------------------------------------


In [857]:
loss_test = evaluate_trigram(W, xs_test, ys_test)
print(f"loss_test: {loss_test:.3f}")

loss_test: 2.392


### Exercise 5: Use Cross Entropy

In [862]:
import random
import torch
import torch.nn.functional as F

names = open('names.txt', 'r').read().splitlines() 

random.seed(42)
g = torch.Generator().manual_seed(2147483647)
random.shuffle(names)

n = len(names)

n_train = int(0.8 * n)
n_dev   = int(0.1 * n)

train_names = names[:n_train]
dev_set = names[n_train:n_train + n_dev]
test_set = names[n_train+n_dev:]



In [863]:
def evaluate_trigram(W, xs, ys):
    logits = W[xs] 
    loss = F.cross_entropy(logits, ys)
    return loss.item()


In [864]:
num_train = xs_train.shape[0]
reg_term = 0.01
W = torch.randn(27*27, 27, requires_grad=True, generator=g)

In [866]:
for _ in range(10000):
    
    logits = W[xs_train] 
    loss = F.cross_entropy(logits, ys_train) + reg_term * (W**2).mean()

    W.grad = None
    loss.backward()
    W.data -= 50 * W.grad
    

train_loss = evaluate_trigram(W, xs_train, ys_train)
dev_loss   = evaluate_trigram(W, xs_dev, ys_dev)

loss_models[reg_term] = dev_loss

print(f"train_loss: {train_loss:.3f} | reg={reg_term}")
print(f"dev_loss:   {dev_loss:.3f} | reg={reg_term}")
print("-" * 40)



train_loss: 2.193 | reg=0.01
dev_loss:   2.222 | reg=0.01
----------------------------------------


In [867]:
loss_test = evaluate_trigram(W, xs_test, ys_test)
print(f"loss_test: {loss_test:.3f}")

loss_test: 2.223


Slightly Overfit

Cross Entropy is better, because it only does the softmax on the desired output, not on all the dataset, so it is actually faster. 